# Bhaashik · NLLB adaptation on Colab / Kaggle

A complete **baseline → finetune → evaluate → error review** workflow for Bhojpuri, Maithili, Magahi, Awadhi, Braj, Nepali and Marwari. Uses a standard HF encoder–decoder, not a chat LLM.

Use the first setup cell with `SETUP_MODE='base'` while preparing data on CPU. For model inference and training, select a GPU runtime (T4 or better), restart the session, and change `SETUP_MODE='train'`; the installer will verify CUDA. Enable Internet in Kaggle and run in order. Free GPU availability and memory are not guaranteed. NLLB weights are CC-BY-NC 4.0. Accept upstream terms when required. Never paste tokens into cells. Example sentences are software fixtures, not research gold.

No notebook has been run on a hosted GPU as part of this repository's offline validation. The local integration tests exercise the same training, export and prediction functions with a tiny random model.

Setup supports Python 3.10–3.13. Python 3.13 uses SentencePiece 0.2.1 wheels; older supported Python versions retain 0.2.0. The shared installer does not install compilers or invoke sudo. Restart the kernel before setup if model libraries have already been imported.


In [ ]:
from pathlib import Path
import os, subprocess, sys
# CPU preparation by default. Change to 'train' when GPU training begins.
SETUP_MODE = 'base'  # base | preflight | train | train,repairs
REQUIRE_CUDA = SETUP_MODE.startswith('train')
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
ROOT = Path('/kaggle/working/BhaashikMTFinetuning' if Path('/kaggle/working').exists() else '/content/BhaashikMTFinetuning')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/bhaashik/BhaashikMTFinetuning.git',str(ROOT)], check=True)
os.chdir(ROOT)
# Use the shared installer: compatible wheels, visible pip errors, no sudo fallback.
installer = ROOT/'scripts/setup_notebook.py'
assert installer.is_file(), 'This checkout is old. Save local work, then refresh/reclone the updated repository.'
loaded = [name for name in ('torch','transformers','sentencepiece','peft','datasets') if name in sys.modules]
if loaded:
    raise RuntimeError('Restart the notebook session/kernel before installation; already imported: '+', '.join(loaded))
command = [sys.executable, str(installer), '--extras', SETUP_MODE]
if REQUIRE_CUDA:
    command.append('--require-cuda')
subprocess.run(command, check=True)
# The active notebook kernel predates pip's new editable-install .pth file.
# Expose this checkout immediately without requiring a second kernel restart.
project_src = str(ROOT / 'src')
if project_src not in sys.path:
    sys.path.insert(0, project_src)
import bhaashik
print('Imported Bhaashik package from:', bhaashik.__file__)
if REQUIRE_CUDA:
    import torch
    print('Notebook GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU data-preparation setup complete; switch to train + GPU for training cells.')


## Choose a direction and data

Choose Bhojpuri (`bho_Deva`), Maithili (`mai_Deva`), Magahi (`mag_Deva`), Awadhi (`awa_Deva`), Nepali (`npi_Deva`), Braj (`bra_Deva`, project-local), or Marwari (`mwr_Deva`, project-local). Hindi (`hin_Deva`) and English (`eng_Latn`) may be bridge languages. Native NLLB does **not** support Braj or Marwari. Check model-specific language inventories before inferring support.

For research set `USE_EXAMPLE_DATA=False`, supply licensed, document-disjoint `train.jsonl`, `dev.jsonl` and `test.jsonl` under `data/local/{PAIR}`, and run the corpus diagnostics before downloading large models. `USE_EXAMPLE_DATA=True` is only a two-example-per-split mechanics test, not a meaningful benchmark.


In [ ]:
import json
SRC, TGT = 'bho_Deva', 'hin_Deva'
SEED, MODE = 42, 'lora'  # Repeat with seeds 13, 42, 87; compare full separately.
MODEL = 'facebook/nllb-200-distilled-600M'
REVISION = None  # Set an immutable Hub commit for final experiments.
USE_EXAMPLE_DATA = True  # Only a mechanics demonstration, not a research result.
PAIR = f"{SRC.split('_')[0]}-{TGT.split('_')[0]}"
if USE_EXAMPLE_DATA and (SRC, TGT) != ('bho_Deva', 'hin_Deva'):
    raise ValueError('Example data are Bhojpuri → Hindi only; supply matching data.')
DATA = Path('data/examples') if USE_EXAMPLE_DATA else Path('data/local') / PAIR
RUN = Path('runs') / f'{PAIR}-{MODE}-s{SEED}'

# Multilingual route controls: raw sentences remain unchanged.
NLLB_CONDITIONING = "native"  # native | dedicated_token | text_prefix
NLLB_PROXY_SOURCE = None  # explicit native donor tag for a text-prefix source proxy
NLLB_PROXY_TARGET = None  # explicit native donor tag for a text-prefix target proxy
EXTEND_NLLB = False  # opt-in initialization of missing registered language tokens; full training required


## Optional single-file train/dev/test splitting (before language migration)

If you have **one parallel JSONL per language pair** (for example `data/local/bho-hin/parallel.jsonl`), set `SPLIT_FROM_SINGLE_FILE=True`. A shared splitter writes deterministic, *document/source-group-disjoint* `train.jsonl`, `dev.jsonl`, and `test.jsonl` files to a separate directory. The seed and train/dev/test ratios are configurable; the original parallel file is never altered. Records sharing a `document_id` or the same normalized source sentence stay together, so actual proportions can differ from requested ratios.

`VALIDATE_DATA=False` is the default for initial mechanics tests: it skips the full corpus diagnostics and figure generation. Basic JSONL format, nonempty splits, and grouped split integrity are still enforced. Set it to `True` for real-data assessment. The model training functions still perform their own mandatory train/dev leakage checks. **Do not use an unvalidated corpus for scientific results.**

**Two aligned `.txt` files:** Set `CONVERT_ALIGNED_TEXT=True` and provide `SOURCE_TEXT_FILE` and `TARGET_TEXT_FILE` (including Kaggle `/kaggle/input/...` paths). Conversion creates tagged `parallel.jsonl` before the seeded splitting function. Both files must contain one sentence per corresponding line, with no blank lines or unequal line counts. It preserves Unicode text, IDs and original files; it cannot verify translation correctness. If `CONVERT_ALIGNED_TEXT=False`, the existing JSONL/split workflow remains unchanged.


In [ ]:
from bhaashik.data_splitting import split_parallel_jsonl
from bhaashik.data import read_jsonl, audit

from bhaashik.text_conversion import convert_aligned_text_files

CONVERT_ALIGNED_TEXT = False
# Both files must have exactly one corresponding sentence per physical line.
# Kaggle datasets can be read directly under /kaggle/input/.
SOURCE_TEXT_FILE = Path("/kaggle/input/my-parallel-data/source.txt")
TARGET_TEXT_FILE = Path("/kaggle/input/my-parallel-data/target.txt")
CONVERTED_JSONL = ROOT / "data" / "local" / PAIR / "parallel.jsonl"

if CONVERT_ALIGNED_TEXT:
    assert not globals().get("USE_EXAMPLE_DATA", False), (
        "Disable example data before importing your own aligned text files.")
    converted = convert_aligned_text_files(
        SOURCE_TEXT_FILE, TARGET_TEXT_FILE, CONVERTED_JSONL,
        source_lang=SRC, target_lang=TGT,
    )
    print("Converted aligned sentence pairs:", converted["sentence_pairs"])
    print("Converted JSONL:", CONVERTED_JSONL)
    print("Original text files remain unchanged.")
    SPLIT_FROM_SINGLE_FILE = True

SINGLE_FILE = CONVERTED_JSONL
SPLIT_SEED = SEED         # Set independently of the training seed if desired.
SPLIT_RATIOS = (0.80, 0.10, 0.10)  # train, dev, test
VALIDATE_DATA = False     # True: full corpus diagnostics and leak audit.
GROUP_BY_DOCUMENT = True

if SPLIT_FROM_SINGLE_FILE:
    assert not USE_EXAMPLE_DATA, "Example fixtures already contain three splits."
    DATA = ROOT / "data" / "local" / PAIR / (
        f"splits-s{SPLIT_SEED}-{int(SPLIT_RATIOS[0]*100)}"
        f"-{int(SPLIT_RATIOS[1]*100)}-{int(SPLIT_RATIOS[2]*100)}"
    )
    split_report = split_parallel_jsonl(
        SINGLE_FILE, DATA, seed=SPLIT_SEED, ratios=SPLIT_RATIOS,
        source_lang=SRC, target_lang=TGT,
        group_by_document=GROUP_BY_DOCUMENT, validate=VALIDATE_DATA,
    )
    print("Split counts:", split_report["counts"])
    print("Actual ratios:", split_report["actual_ratios"])

DATA_PATHS = {part: DATA / f"{part}.jsonl" for part in ("train", "dev", "test")}
for part, path in DATA_PATHS.items():
    assert path.is_file(), f"Missing {part} data: {path}"
    print(part, len(read_jsonl(path)), "pairs")
if VALIDATE_DATA:
    print("Exact train/dev/test audit:", audit(DATA_PATHS))
else:
    print("Extended corpus validation is OFF (initial test mode).")


## Explicit per-sentence language tags (optional migration)

If the three existing splits have no language metadata, generate **new tagged copies** using their selected single language direction. For mixed-language corpora, supply correct `source_lang` and `target_lang` on **every row** yourself; never label an entire mixed file as a single language. The raw `source`/`target` strings must remain unprefixed. Every pipeline stage retains these fields. Existing clean legacy JSONL remains accepted with notebook-level `SRC`/`TGT` defaults.

For unsupported NLLB languages, select either `text_prefix` with **explicit native proxy tags**, or `dedicated_token` with `EXTEND_NLLB=True` and `MODE='full'`. A new language token is initialized, not learned, until finetuning. See `docs/multilingual-adaptation.md`.


In [ ]:
from bhaashik.multilingual import write_tagged_copy, inspect_routes, require_routes
MIGRATE_LEGACY_SPLITS = False
if MIGRATE_LEGACY_SPLITS:
    tagged_dir = ROOT / "data" / "local" / f"{PAIR}-tagged"
    for split in ("train", "dev", "test"):
        source = DATA / f"{split}.jsonl"
        target = tagged_dir / f"{split}.jsonl"
        if not target.exists():
            write_tagged_copy(source, target, SRC, TGT)
    DATA = tagged_dir
for split in ("train", "dev", "test"):
    _, routes = require_routes(DATA / f"{split}.jsonl", SRC, TGT)
    print(split, routes)
if EXTEND_NLLB:
    assert SETUP_MODE.startswith("train"), "Enable GPU training for language-token extension."
    assert MODE == "full" and NLLB_CONDITIONING == "dedicated_token", (
        "New language-token embeddings require full finetuning.")
    from bhaashik.language_support import LANGUAGE_BY_TAG
    from bhaashik.modeling import extend
    missing = sorted({
        code for split in ("train", "dev")
        for row in require_routes(DATA / f"{split}.jsonl", SRC, TGT)[0]
        for code in (row.get("source_lang", SRC), row.get("target_lang", TGT))
        if code in LANGUAGE_BY_TAG and not LANGUAGE_BY_TAG[code].nllb_native
    })
    for new_tag in missing:
        extended_model = ROOT / "runs" / f"nllb-language-extension-{new_tag}"
        if not (extended_model / "bhaashik_extension.json").exists():
            extend(MODEL, extended_model, new_tag, donor="hin_Deva", revision=REVISION)
        MODEL = str(extended_model)
        REVISION = None
    print("Extended checkpoint:", MODEL, "new tags:", missing)


## Optional pre-training corpus validation and visualizations

With `VALIDATE_DATA=True`, inspect train/dev/test leakage, sentence-length distributions, scripts, provenance, near-duplicates and missing metadata before real-data training. With `False`, skip the expensive analysis while testing the mechanics. **Validation is separate from the normal development-set evaluation performed during training.** Never treat an unvalidated split as research-ready.


In [ ]:
from bhaashik.pretrain_review import inspect_splits, plot_review
from bhaashik.language_support import language_options, require_native_nllb

REVIEW_DIR = ROOT / "runs" / f"{PAIR}-pretrain-review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print("Language coverage (native != paper claim):")
for item in language_options():
    print(item)
if VALIDATE_DATA:
    CORPUS_REVIEW = inspect_splits(DATA_PATHS, REVIEW_DIR)
    print("Data audit ready for review:", CORPUS_REVIEW["ready_for_training"])
    for issue in CORPUS_REVIEW["issues"]:
        print(issue["severity"].upper(), issue["check"], issue["message"])
    for chart in plot_review(REVIEW_DIR):
        print("Saved figure:", chart)
        try:
            from IPython.display import display, Image
            display(Image(filename=str(chart)))
        except ImportError:
            pass
else:
    CORPUS_REVIEW = {"ready_for_training": False, "issues": [],
                     "validation_skipped": True}
    print("Full corpus diagnostics and charts skipped. Set VALIDATE_DATA=True for real-data review.")


## Unfinetuned baselines on development data

First freeze a reproducible sample of **development** rows (never test rows). Run the same sample through NLLB and any optional baseline models. Model downloads may take time; set `RUN_NLLB_DEV_BASELINE=False` during CPU-only corpus review. Native NLLB is available for Bhojpuri, Maithili, Magahi, Awadhi and Nepali, but not Braj or Marwari; unsupported directions require an explicitly trained language extension. Maithili/Nepali are supported in IndicTrans2. BhashaVerse 36-language claims exceed the checkpoint's explicit Hub language list, so the adapter permits only card-verified languages.


In [ ]:
from bhaashik.model_comparisons import select_dev_sample
from bhaashik.language_support import require_native_nllb, LANGUAGE_BY_TAG

DEV_SAMPLE_SIZE = 100   # small, fixed development sample for cross-model comparisons
DEV_SAMPLE = REVIEW_DIR / "dev_comparison_sample.jsonl"
if not DEV_SAMPLE.exists():
    select_dev_sample(DATA / "dev.jsonl", DEV_SAMPLE, sample_size=DEV_SAMPLE_SIZE, seed=SEED)
else:
    from bhaashik.data import file_hash
    sample_manifest = json.loads(Path(str(DEV_SAMPLE) + ".manifest.json").read_text())
    assert sample_manifest["source_sha256"] == file_hash(DATA / "dev.jsonl"), (
        "Development data changed; use a new run/review directory and sample"
    )
print("Frozen dev sample:", DEV_SAMPLE)
ALL_REVIEW_TAGS = {
    code for split in ("train", "dev")
    for row in require_routes(DATA / f"{split}.jsonl", SRC, TGT)[0]
    for code in (row.get("source_lang", SRC), row.get("target_lang", TGT))
}
UNSUPPORTED_TAGS = {
    code for code in ALL_REVIEW_TAGS
    if code in LANGUAGE_BY_TAG and not LANGUAGE_BY_TAG[code].nllb_native
}
UNSUPPORTED_SOURCES = {
    row.get("source_lang", SRC) for split in ("train", "dev")
    for row in require_routes(DATA / f"{split}.jsonl", SRC, TGT)[0]
    if row.get("source_lang", SRC) in UNSUPPORTED_TAGS
}
UNSUPPORTED_TARGETS = {
    row.get("target_lang", TGT) for split in ("train", "dev")
    for row in require_routes(DATA / f"{split}.jsonl", SRC, TGT)[0]
    if row.get("target_lang", TGT) in UNSUPPORTED_TAGS
}
NLLB_SUPPORTED = (
    not UNSUPPORTED_TAGS
    or (NLLB_CONDITIONING == "text_prefix"
        and (not UNSUPPORTED_SOURCES or NLLB_PROXY_SOURCE is not None)
        and (not UNSUPPORTED_TARGETS or NLLB_PROXY_TARGET is not None))
    or (NLLB_CONDITIONING == "dedicated_token" and EXTEND_NLLB)
)
RUN_NLLB_DEV_BASELINE = SETUP_MODE.startswith("train") and NLLB_SUPPORTED
DEV_PREDICTIONS = {}
if RUN_NLLB_DEV_BASELINE:
    from bhaashik.modeling import translate
    NLLB_DEV = REVIEW_DIR / "nllb-dev.jsonl"
    if not NLLB_DEV.exists():
        translate(MODEL, SRC, TGT, DEV_SAMPLE, NLLB_DEV, revision=REVISION,
                  conditioning=NLLB_CONDITIONING, proxy_source_lang=NLLB_PROXY_SOURCE,
                  proxy_target_lang=NLLB_PROXY_TARGET,
                  batch_size=1, max_length=128, max_new_tokens=128, beams=4)
    DEV_PREDICTIONS["NLLB"] = NLLB_DEV
else:
    print("NLLB dev baseline skipped (CPU review mode or unsupported native language tag).")


## Optional IndicTrans2 and BhashaVerse comparators

Both are **off by default** and use the exact frozen dev sample. IndicTrans2 requires `IndicTransToolkit`, supported native tags and a separate upstream gated-model/custom-code installation if applicable; inspect the model code before enabling `ALLOW_INDIC_REMOTE_CODE`. BhashaVerse requires a previously downloaded and reviewed `hf_inference.py` from the model's Hugging Face repository, with its required dictionary, SentencePiece and weights. It must **not** use the standard MBart/AutoTokenizer path. The model adapters write prediction JSONL files with matching IDs for fair scoring. Do not install their incompatible dependency stacks into an already-running NLLB experiment without validating compatibility; outputs can be generated in separate Kaggle/Colab sessions and supplied as files here.


In [ ]:
from bhaashik.model_comparisons import (
    run_indictrans2_dev, run_bhashaverse_dev,
)
from bhaashik.language_support import LANGUAGE_BY_TAG

RUN_INDICTRANS2 = False
ALLOW_INDIC_REMOTE_CODE = False
RUN_BHASHAVERSE = False
BHASHAVERSE_INFERENCE_SCRIPT = None  # Path to reviewed upstream hf_inference.py
# Optional: add existing externally generated predictions here:
EXTERNAL_DEV_PREDICTIONS = {}  # e.g. {"IndicTrans2": REVIEW_DIR / "indictrans2-dev.jsonl"}

if RUN_INDICTRANS2:
    output = REVIEW_DIR / "indictrans2-dev.jsonl"
    if not output.exists():
        run_indictrans2_dev(
            DEV_SAMPLE, output, SRC, TGT,
            allow_remote_code=ALLOW_INDIC_REMOTE_CODE, batch_size=4,
        )
    DEV_PREDICTIONS["IndicTrans2"] = output

if RUN_BHASHAVERSE:
    if BHASHAVERSE_INFERENCE_SCRIPT is None:
        raise ValueError("Provide a reviewed upstream hf_inference.py file path")
    output = REVIEW_DIR / "bhashaverse-dev.jsonl"
    if not output.exists():
        run_bhashaverse_dev(
            DEV_SAMPLE, output, SRC, TGT,
            inference_script=BHASHAVERSE_INFERENCE_SCRIPT, batch_size=4,
        )
    DEV_PREDICTIONS["BhashaVerse"] = output

DEV_PREDICTIONS.update(EXTERNAL_DEV_PREDICTIONS)
print("Development predictions to compare:", DEV_PREDICTIONS)


## Pre-training comparison and approval gate

Generate a reproducible development-set score table and charts (sacreBLEU, chrF++, review flags). Predictions from every model must match the **same IDs, source, reference and metadata**; otherwise scoring refuses to compare them. Small samples are only smoke tests, not scientific evidence. Review per-language/script quality and the diagnostics before training. For real data, explicitly set `APPROVE_REAL_TRAINING=True` **after** validation; this prevents accidental finetuning during exploration. The held-out test set is reserved until after training.


In [ ]:
from bhaashik.pretrain_review import compare_dev, plot_review
if DEV_PREDICTIONS:
    DEV_REVIEW = compare_dev(DEV_SAMPLE, DEV_PREDICTIONS, REVIEW_DIR)
    for model_result in DEV_REVIEW["models"]:
        print(model_result["model"], "chrF++", model_result["chrfpp"],
              "BLEU", model_result["bleu"], "flags", model_result["review_flags"])
    for chart in plot_review(REVIEW_DIR):
        print("Saved figure:", chart)
        try:
            from IPython.display import display, Image
            display(Image(filename=str(chart)))
        except ImportError:
            pass
else:
    print("No baseline predictions yet; data diagnostics are available above.")

APPROVE_REAL_TRAINING = False  # Set True only after reviewing real data
TRAIN_APPROVED = SETUP_MODE.startswith("train") and NLLB_SUPPORTED and (
    USE_EXAMPLE_DATA or (
        APPROVE_REAL_TRAINING
        and (not VALIDATE_DATA or CORPUS_REVIEW["ready_for_training"])
        and bool(DEV_PREDICTIONS)
    )
)
print("Training approved:", TRAIN_APPROVED)


## Finetune and export a normal HF checkpoint (requires approval for real data)

Set `APPROVE_REAL_TRAINING=True` in the earlier review cell only after checking diagnostics and native model coverage. The default LoRA configuration uses batch 1, gradient accumulation 16, and sequence length 128. For resumability preserve `RUN/checkpoints/checkpoint-N` separately; the fast final-results archive excludes Trainer snapshots unless requested. Use a fresh run directory per seed/model/data split.


In [ ]:
if TRAIN_APPROVED:
    from bhaashik.modeling import train
    config = dict(model=MODEL, revision=REVISION, src_lang=SRC, tgt_lang=TGT,
                  train_file=str(DATA/'train.jsonl'), dev_file=str(DATA/'dev.jsonl'), output_dir=str(RUN),
                  mode=MODE, seed=SEED, epochs=3, learning_rate=2e-4 if MODE=='lora' else 2e-5,
                  batch_size=1, gradient_accumulation_steps=16, max_length=128,
                  precision='fp16', gradient_checkpointing=True, lora_rank=8,
                  conditioning=NLLB_CONDITIONING)
    if NLLB_PROXY_SOURCE is not None:
        config["proxy_source_lang"] = NLLB_PROXY_SOURCE
    if NLLB_PROXY_TARGET is not None:
        config["proxy_target_lang"] = NLLB_PROXY_TARGET
    config_path = Path('configs') / f'{PAIR}-{MODE}-s{SEED}.json'
    config_path.write_text(json.dumps(config, indent=2))
    print(train(config_path))
    (RUN / "training_config.json").write_text(json.dumps(config, indent=2))
else:
    print("Skipped training. Complete dev review and explicitly approve real-data training.")


In [ ]:
if TRAIN_APPROVED:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    # LoRA is merged; no PEFT wrapper/custom architecture is needed for consumers.
    tokenizer = AutoTokenizer.from_pretrained(RUN)
    model = AutoModelForSeq2SeqLM.from_pretrained(RUN)
    print(type(model).__name__, len(tokenizer))
    del model, tokenizer
    import gc
    gc.collect(); torch.cuda.empty_cache()
    translate(RUN, SRC, TGT, DATA/'test.jsonl', RUN/'predictions.jsonl',
              conditioning=NLLB_CONDITIONING, proxy_source_lang=NLLB_PROXY_SOURCE,
              proxy_target_lang=NLLB_PROXY_TARGET,
              batch_size=1, max_length=128, max_new_tokens=128, beams=4)
else:
    print("Skipped post-training translation; real-data training not approved.")


## Held-out test evaluation after training

This is the first point where the frozen **test** split is translated. It runs the untouched NLLB baseline and then compares the trained system with the same IDs/references. Do not inspect test scores while selecting training configurations or choosing hyperparameters.


In [ ]:
if TRAIN_APPROVED:
    from bhaashik.modeling import translate
    BASE = Path('runs') / f'{PAIR}-baseline'
    baseline_path = BASE / 'predictions.jsonl'
    if not baseline_path.exists():
        translate(MODEL, SRC, TGT, DATA/'test.jsonl', baseline_path, revision=REVISION,
                  conditioning=NLLB_CONDITIONING, proxy_source_lang=NLLB_PROXY_SOURCE,
                  proxy_target_lang=NLLB_PROXY_TARGET,
                  batch_size=1, max_length=128, max_new_tokens=128, beams=4)
    print("Held-out test baseline:", baseline_path)
else:
    BASE = Path('runs') / f'{PAIR}-baseline'
    print("Test baseline not run: real-data training not approved.")


## Paired evaluation and human review

Corpus BLEU/chrF++ are not dialect-fidelity judgments. Bootstrap whole documents when IDs exist. A fixture-sized sample cannot support a paper claim; add qualified native-speaker annotation and multiple training seeds. Exported flags are triage hints, not automatic linguistic diagnoses. Import `evaluation.json` in the Pages portal to view actual scores.

In [ ]:
if TRAIN_APPROVED:
    from bhaashik.evaluation import evaluate, export_errors
    result = evaluate(RUN/'predictions.jsonl', RUN/'evaluation.json',
                      baseline=BASE/'predictions.jsonl', samples=1000, seed=SEED)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    export_errors(RUN/'predictions.jsonl', RUN/'annotations.csv', seed=SEED)
else:
    print("Final test evaluation skipped (no trained model).")


## Preserve artifacts

The fast archive stores **only the current experiment**, the corresponding baseline and the training configuration. It preserves the exported HF model/tokenizer, predictions, manifests, metrics and annotations. By default it **excludes the large Trainer checkpoint directory**, so it is suitable for preserving final results but **cannot be used to resume training**. Set `INCLUDE_CHECKPOINTS=True` if you require optimizer/scheduler checkpoint state for resuming. ZIP files use no additional compression and can therefore be larger, but are quicker to create; model weights may already be internally efficient. All outputs remain under the same established repository structure. Dataset contents are not copied into this archive; preserve your licensed data separately. Save the Kaggle version with outputs or copy/download the ZIP before ending the session.


In [ ]:
if TRAIN_APPROVED:
    from bhaashik.artifacts import archive_experiment
    
    # False: fast final-results archive; True: include optimizer/trainer checkpoints for resuming
    INCLUDE_CHECKPOINTS = False
    archive_path = ROOT.parent / f'bhaashik-{PAIR}-{MODE}-s{SEED}-artifacts.zip'
    archive = archive_experiment(
        ROOT, RUN, archive_path, baseline_dir=BASE, config_path=config_path,
        include_checkpoints=INCLUDE_CHECKPOINTS, report_dir=REVIEW_DIR,
    )
    print('Download or copy to persistent storage:', archive)
    print('Size (GiB):', round(archive.stat().st_size / 2**30, 2))
    # Colab: from google.colab import files; files.download(str(archive))
    # Kaggle: use the notebook Output files panel.
else:
    print("No trained run to archive. Pretraining review reports are in:", REVIEW_DIR)
